### LRU Cache

In [1]:
class Node:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.next = None
        self.prev = None

class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = {}
        self.head = Node(0, 0)
        self.tail = Node(0, 0)
        self.head.next = self.tail
        self.tail.prev = self.head

    def get(self, key):
        if key not in self.cache.keys():
            return -1
        node = self.cache[key]
        self._remove(node)
        self._add(node)
        return node.value

    def put(self, key, value):
        if key in self.cache.keys():
            node = self.cache[key]
            self._remove(node)
        new_node = Node(key, value)
        self._add(new_node)
        self.cache[key] = new_node
        if len(self.cache.keys()) > self.capacity:
            lru = self.head.next
            self._remove(lru)
            del self.cache[lru.key]

    def _remove(self, node):
        prev = node.prev
        next = node.next
        prev.next = next
        next.prev = prev

    def _add(self, node):
        prev = self.tail.prev
        prev.next = node
        node.prev = prev
        node.next = self.tail
        self.tail.prev = node

### TTL Cache

In [1]:
import time

class Item:
    def __init__(self, value, expiry):
        self.value = value
        self.expiry = expiry

class TTLCache:
    def __init__(self):
        self.cache = {}

    def get(self, key):
        self.update()
        return self.cache[key].value if key in self.cache.keys() else -1

    def put(self, key, value, duration):
        self.update()
        item = Item(value, time.time() + duration)
        self.cache[key] = item

    def display(self):
        self.update()
        return {key: item.value for key, item in self.cache.items()}
    
    def update(self):
        keys_to_delete = [key for key in self.cache.keys() if self.cache[key].expiry <= time.time()]
        for key in keys_to_delete:
            del self.cache[key]

In [2]:
cache = TTLCache()
cache.put("key1", "value1", 5)
cache.put("key2", "stan", 15)
cache.put("key3", "abi", 2)
cache.put("key4", "dave", 10)
print(cache.display())
print(cache.get("key1"))
print(cache.get("key2"))
time.sleep(6)
print(cache.get("key1"))
print(cache.display())

{'key1': 'value1', 'key2': 'stan', 'key3': 'abi', 'key4': 'dave'}
value1
stan
-1
{'key2': 'stan', 'key4': 'dave'}


### Unix File Search API

In [25]:
import os

class SearchCriterion:
    def match(self, file):
        raise NotImplementedError("This method should be overridden by subclass")
    
class ExtensionCriterion(SearchCriterion):
    def __init__(self, extension):
        self.extension = extension

    def match(self, file):
        return file.endswith(self.extension)
    
class NameCriterion(SearchCriterion):
    def __init__(self, name):
        self.name = name

    def match(self, file):
        return self.name in file
    
class SizeCriterion(SearchCriterion):
    def __init__(self, min_size, max_size=float('inf')):
        self.min_size = min_size
        self.max_size = max_size

    def match(self, file):
        size = os.path.getsize(file)
        return self.min_size <= size <= self.max_size
    
class ComposeCriterion(SearchCriterion):
    def __init__(self, criteria):
        self.criteria = criteria

    def match(self, file):
        return all(criterion.match(file) for criterion in self.criteria)
    
class AndCriterion(SearchCriterion):
    def __init__(self, *criteria):
        self.criteria = criteria

    def match(self, file):
        return all(criterion.match(file) for criterion in self.criteria)

class OrCriterion(SearchCriterion):
    def __init__(self, *criteria):
        self.criteria = criteria

    def match(self, file):
        return any(criterion.match(file) for criterion in self.criteria)
    
class FileSearchAPI:
    def search(self, start_path, criterion):
        matched_files = []
        for root, dirs, files in os.walk(start_path):
            for file in files:
                full_path = os.path.join(root, file)
                if criterion.match(full_path):
                    matched_files.append(full_path)
        return matched_files

In [26]:
compose_criterion = ComposeCriterion([
    ExtensionCriterion('.png'),
    SizeCriterion(1024)
])

api = FileSearchAPI()
matched_files = api.search('/Users/osd/Work/', compose_criterion)

print(matched_files)

size_criterion = SizeCriterion(5)
extension_txt_criterion = ExtensionCriterion('.txt')
extension_jpg_criterion = ExtensionCriterion('.jpg')

and_criterion = AndCriterion(extension_txt_criterion, size_criterion)
or_criterion = OrCriterion(and_criterion, extension_jpg_criterion)

api = FileSearchAPI()
matched_files = api.search('/Users/osd/Work/', or_criterion)

print(matched_files)

['/Users/osd/Work/Computational Photography Lab/cplab_logo_glow.png', '/Users/osd/Work/Computational Photography Lab/logo-long.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/wine-3900655_1920.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/9422601_114626567000_2.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/h_90.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/h_84.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/h_217.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/t_22.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/h_53.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/h_47.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/t_36.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/glass-783678_1920.png', '/Users/osd/Work/Computational Photography Lab/Dataset/fg/wine-glasses-2300713_1280_1.png', '/Users/osd/Work/Computational Photograp

### In-Memory File Sytem

In [185]:
from collections import deque

class FileSystemEntity:
    def __init__(self, name, path):
        self.name = name
        self.path = path

class File(FileSystemEntity):
    def __init__(self, name, path, content=''):
        super().__init__(name, path)
        self.content = content

class Directory(FileSystemEntity):
    def __init__(self, name, path):
        super().__init__(name, path)
        self.children = {}

class FileSystem:
    def __init__(self):
        self.root = Directory('root', '/')
        self.cwd = self.root

    def navigate(self, path, create=False, isFile=False, content=''):
        current = self.root if path.startswith('/') else self.cwd
        parts = [p for p in path.split('/') if p]
        location = ''
        for i, part in enumerate(parts):
            if part not in current.children:
                if create:
                    location += f'/{part}'
                    current.children[part] = File(part, location, content) if isFile and (i == len(parts) - 1) else Directory(part, location)
                else:
                    raise FileNotFoundError('The specified path does not exist')
            current = current.children[part]
            if isinstance(current, File) and i != len(parts) - 1:
                raise Exception('Cannot navigate through a file.')
        if isFile and not isinstance(current, File):
            raise Exception('The specified path does not lead to a file.')
        elif not isFile and not isinstance(current, Directory):
            raise Exception('The specified path does not lead to a directory.')
        return current

    def mkdir(self, path):
        _ = self.navigate(path, create=True)

    def touch(self, path, content=''):
        _ = self.navigate(path, create=True, isFile=True, content=content)

    def read(self, path):
        file = self.navigate(path, isFile=True)
        return file.content

    def write(self, path, content):
        file = self.navigate(path, isFile=True)
        file.content = content

    def ls(self, path=None):
        if path == None:
            path = self.cwd.path
        directory = self.navigate(path)
        return [directory.children[child].name for child in directory.children]

    def cd(self, path):
        directory = self.navigate(path)
        self.cwd = directory

    def rm(self, path=None):
        if path == None:
            path = self.cwd.path
        parts = path.split('/')
        name = parts.pop()
        parent_path = '/'.join(parts)
        parent_directory = self.navigate(parent_path)
        if name not in parent_directory.children:
            raise FileNotFoundError('The specified path does not exist.')
        del parent_directory.children[name]

    def walk(self, path):
        start = self.navigate(path, create=True)
        queue = deque([start])
        visited = [start.path]
        while queue:
            vertex = queue.popleft()
            if isinstance(vertex, Directory):
                children = vertex.children
                for part in children:
                    child = children[part]
                    visited.append(child.path)
                    queue.append(child)
        return visited

In [186]:
system = FileSystem()
system.mkdir('obum')
system.touch('obum.txt', 'I love myself very much!')
system.touch('/kenechi/masters/thesis/final.pdf')
print(system.ls())

['obum', 'obum.txt', 'kenechi']


In [187]:
print(system.walk('/'))

['/', '/obum', '/obum.txt', '/kenechi', '/kenechi/masters', '/kenechi/masters/thesis', '/kenechi/masters/thesis/final.pdf']


In [188]:
print(system.read('obum.txt'))
system.cd('/kenechi')
print(system.ls('masters'))
system.write('masters/thesis/final.pdf', 'This is Kenechi Thesis')

I love myself very much!
['thesis']


In [189]:
print(system.read('/kenechi/masters/thesis/final.pdf'))
system.rm('masters/thesis')
print(system.walk('/'))

This is Kenechi Thesis
['/', '/obum', '/obum.txt', '/kenechi', '/kenechi/masters']
